# FF1: Konsolidierung der Lebensstil-Landschaft

In [1]:
source("setup.R")

data.table 1.17.8 using 8 threads (see ?getDTthreads).  Latest news: r-datatable.com

Attaching package: ‘igraph’

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union


Attaching package: ‘dbscan’

The following object is masked from ‘package:stats’:

    as.dendrogram

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%--%()      masks igraph::%--%()
✖ dplyr::as_data_frame() masks tibble::as_data_frame(), igraph::as_data_frame()
✖ dplyr::between()       masks data.table::between()
✖ purrr::compose()       masks igraph::compose()
✖ tidyr::crossing()      masks igraph::crossing()
✖ dplyr::filte

Warning messages:
1: package ‘igraph’ was built under R version 4.5.3 
2: package ‘ineq’ was built under R version 4.5.2 


In [2]:
trend <- function(tab, masse, zeit = "jahr") {
  for (m in masse) {
    cat(sprintf("  %-14s tau = %+.2f\n", m,
                cor(tab[[zeit]], tab[[m]], method = "kendall")))
  }
}

## Netzwerk und Module

In [3]:
build_graph <- function(jahr, subs, k = 15) {
  v <- fread(vek_datei(jahr), skip = 1, header = FALSE)
  setnames(v, 1, "sub")
  v <- v[sub %in% subs]
  M <- as.matrix(v[, -1])
  rownames(M) <- v$sub
  M <- M / sqrt(rowSums(M^2))

  kn   <- dbscan::kNN(M, k = k)
  n    <- nrow(M)
  from <- rep(seq_len(n), each = k)
  to   <- as.vector(t(kn$id))
  w    <- rowSums(M[from, ] * M[to, ])

  g <- graph_from_data_frame(
    data.frame(from = rownames(M)[from], to = rownames(M)[to], weight = w),
    directed = FALSE)
  igraph::simplify(g, edge.attr.comb = "first")
}

In [4]:
# ab5 und unter5 sind die Gegenprobe auf die Untergrenze: der
# Schwellengraph unten zählt nur Module ab fünf Knoten, dieser Lauf alle.
ff1_roh <- rbindlist(lapply(JAHRE, function(j) {
  g  <- build_graph(j, ana_subs, k = 15)
  wt <- cluster_walktrap(g, weights = E(g)$weight)
  sz <- sort(as.integer(table(membership(wt))), decreasing = TRUE)
  N  <- sum(sz)
  data.table(
    jahr       = j,
    density    = edge_density(g),
    knoten     = vcount(g),
    kanten     = ecount(g),
    komp       = components(g)$no,
    module     = length(wt),
    Q          = modularity(g, membership(wt), weights = E(g)$weight),
    top1       = sz[1] / N,
    top2       = sum(sz[1:2]) / N,
    rosenbluth = ineq::conc(sz, type = "Rosenbluth"),
    ab5        = sum(sz >= 5),
    unter5     = sum(sz <  5),
    groesstes  = sz[1])
}))

ff1 <- ff1_roh[, .(jahr, density, knoten, komp, module, Q, top1, top2, rosenbluth)]
print(ff1)

cat("\nKendalls tau:\n")
trend(ff1, c("module", "Q", "top2", "rosenbluth"))

cat(sprintf("\nModule unter fuenf Knoten ueber alle Jahre: %d\n", sum(ff1_roh$unter5)))

readr::write_csv(ff1, file.path(OUT_DIR, "ff1_knn_masse.csv"))
readr::write_csv(ff1_roh[, .(jahr, knoten, kanten, module, ab5, unter5, groesstes, Q)],
                 file.path(OUT_DIR, "splitter_knn.csv"))

    jahr     density knoten  komp module         Q      top1      top2
   <int>       <num>  <num> <num>  <int>     <num>     <num>     <num>
1:  2016 0.001616004  14412     1    115 0.7215562 0.1421732 0.2784485
2:  2017 0.001575164  14412     1     92 0.7358848 0.2262004 0.3028726
3:  2018 0.001541538  14412     1     86 0.7610131 0.1801277 0.2745629
4:  2019 0.001518600  14412     1     88 0.7778242 0.1713156 0.2474327
5:  2020 0.001499735  14412     1     93 0.7839210 0.1837358 0.2422287
6:  2021 0.001494015  14412     1     85 0.7821560 0.2112129 0.2830974
7:  2022 0.001498560  14412     1     96 0.7819463 0.1708299 0.2140577
8:  2023 0.001510318  14412     1     89 0.7867949 0.1770747 0.2824036
9:  2024 0.001511137  14412     1     87 0.7933314 0.1480711 0.2725507
   rosenbluth
        <num>
1: 0.02484493
2: 0.03202880
3: 0.03327776
4: 0.03080738
5: 0.02936143
6: 0.03296100
7: 0.02669749
8: 0.03075991
9: 0.03222517

Kendalls tau:
  module         tau = -0.22
  Q              tau 

## Robustheit über Nachbarzahl und Schrittlänge

In [5]:
.graphen <- new.env(parent = emptyenv())

get_graph <- function(jahr, k) {
  key <- paste(jahr, k, sep = "_")
  if (is.null(.graphen[[key]])) .graphen[[key]] <- build_graph(jahr, ana_subs, k = k)
  .graphen[[key]]
}

ff1_trend <- function(k, steps) {
  roh <- rbindlist(lapply(JAHRE, function(j) {
    g  <- get_graph(j, k)
    wt <- cluster_walktrap(g, weights = E(g)$weight, steps = steps)
    sz <- sort(as.integer(table(membership(wt))), decreasing = TRUE)
    N  <- sum(sz)
    data.table(k = k, steps = steps, jahr = j,
               module     = length(wt),
               Q          = modularity(g, membership(wt), weights = E(g)$weight),
               top1       = sz[1] / N,
               top2       = sum(sz[1:2]) / N,
               rosenbluth = ineq::conc(sz, type = "Rosenbluth"))
  }))
  tau <- data.table(k = k, steps = steps,
    tau_module     = cor(roh$jahr, roh$module,     method = "kendall"),
    tau_Q          = cor(roh$jahr, roh$Q,          method = "kendall"),
    tau_top2       = cor(roh$jahr, roh$top2,       method = "kendall"),
    tau_rosenbluth = cor(roh$jahr, roh$rosenbluth, method = "kendall"))
  list(roh = roh, tau = tau)
}

In [6]:
gitter   <- CJ(k = c(10, 15, 25, 50), steps = 3:8)
ergebnis <- Map(ff1_trend, gitter$k, gitter$steps)
grid_tau <- rbindlist(lapply(ergebnis, function(x) x$tau))
grid_roh <- rbindlist(lapply(ergebnis, function(x) x$roh))

cat("== Trend ueber die Nachbarzahl, Schrittlaenge vier ==\n")
print(grid_tau[steps == 4])

cat("\n== Trend ueber das ganze Gitter ==\n")
print(grid_tau)

cat(sprintf("\ntau_Q  min %+.2f | median %+.2f | max %+.2f  (%d Kombinationen)\n",
    min(grid_tau$tau_Q), median(grid_tau$tau_Q), max(grid_tau$tau_Q), nrow(grid_tau)))

readr::write_csv(grid_tau, file.path(OUT_DIR, "ff1_robustheit_grid_tau.csv"))
readr::write_csv(grid_roh, file.path(OUT_DIR, "ff1_robustheit_grid_roh.csv"))

== Trend ueber die Nachbarzahl, Schrittlaenge vier ==
       k steps  tau_module     tau_Q   tau_top2 tau_rosenbluth
   <num> <int>       <num>     <num>      <num>          <num>
1:    10     4  0.14085904 0.9444444 -0.6111111    -0.38888889
2:    15     4 -0.22222222 0.8333333 -0.2777778     0.05555556
3:    25     4  0.27777778 0.5555556 -0.5000000    -0.33333333
4:    50     4  0.08451543 0.6111111 -0.2777778    -0.05555556

== Trend ueber das ganze Gitter ==
Index: <steps>
        k steps  tau_module     tau_Q   tau_top2 tau_rosenbluth
    <num> <int>       <num>     <num>      <num>          <num>
 1:    10     3 -0.22222222 0.8888889 -0.1666667     0.00000000
 2:    10     4  0.14085904 0.9444444 -0.6111111    -0.38888889
 3:    10     5  0.08451543 0.8333333 -0.5000000    -0.16666667
 4:    10     6  0.02817181 0.9444444 -0.3888889    -0.11111111
 5:    10     7  0.11433239 0.8333333 -0.2222222    -0.50000000
 6:    10     8  0.11111111 0.7222222  0.0000000    -0.16666667
 7:  

## Messung ohne Partition

In [7]:
set.seed(42)
S_PAARE <- 300000L

ff1_geo <- rbindlist(lapply(JAHRE, function(j) {
  v <- fread(vek_datei(j), skip = 1, header = FALSE)
  setnames(v, 1, "sub")
  v <- v[sub %in% ana_subs]
  M <- as.matrix(v[, -1])
  M <- M / sqrt(rowSums(M^2))
  n <- nrow(M)

  # Allgemeines Niveau des Raumes aus Zufallspaaren
  a  <- sample(n, S_PAARE, replace = TRUE)
  b  <- sample(n, S_PAARE, replace = TRUE)
  ok <- a != b
  gcos <- rowSums(M[a[ok], ] * M[b[ok], ])

  # Nachbarschaftsniveau, cos = 1 - d^2/2 bei Länge eins
  ncos <- 1 - (as.vector(dbscan::kNN(M, k = 15)$dist)^2) / 2

  # Auf wie viele Richtungen verteilt sich die Streuung
  lam   <- svd(scale(M, center = TRUE, scale = FALSE), nu = 0, nv = 0)$d^2
  PR    <- sum(lam)^2 / sum(lam^2)
  dim90 <- which(cumsum(lam) / sum(lam) >= 0.90)[1]

  data.table(jahr = j,
             mean_global = mean(gcos),
             mean_nn     = mean(ncos),
             kontrast    = mean(ncos) - mean(gcos),
             q90_global  = as.numeric(quantile(gcos, 0.90)),
             q10_global  = as.numeric(quantile(gcos, 0.10)),
             PR          = PR,
             dim90       = dim90)
}))
print(ff1_geo)

cat("\nKendalls tau:\n")
trend(ff1_geo, c("mean_global", "mean_nn", "kontrast", "q90_global",
                 "q10_global", "PR", "dim90"))

readr::write_csv(ff1_geo, file.path(OUT_DIR, "ff1_geometrie.csv"))

    jahr mean_global   mean_nn  kontrast q90_global q10_global       PR dim90
   <int>       <num>     <num>     <num>      <num>      <num>    <num> <int>
1:  2016   0.2595452 0.6298390 0.3702937  0.3784450  0.1507989 96.64384   123
2:  2017   0.2495082 0.6308040 0.3812959  0.3562801  0.1480073 94.85766   123
3:  2018   0.2439037 0.6395283 0.3956246  0.3437163  0.1462914 94.89976   122
4:  2019   0.2437699 0.6575925 0.4138226  0.3423806  0.1472510 93.93635   120
5:  2020   0.2469286 0.6738017 0.4268731  0.3436884  0.1518656 91.61622   119
6:  2021   0.2489633 0.6812220 0.4322587  0.3464953  0.1536756 88.86559   118
7:  2022   0.2478673 0.6870491 0.4391818  0.3462480  0.1514905 85.98687   119
8:  2023   0.2519784 0.6923238 0.4403454  0.3529163  0.1540486 83.40776   119
9:  2024   0.2481572 0.6944333 0.4462762  0.3509554  0.1488357 83.26257   119

Kendalls tau:
  mean_global    tau = +0.00
  mean_nn        tau = +1.00
  kontrast       tau = +1.00
  q90_global     tau = -0.06
  q10_globa

## Worauf der Rückgang der Streuungsbreite beruht

In [8]:
panel_check <- rbindlist(lapply(JAHRE, function(j) {
  nm <- fread(vek_datei(j), select = 1, skip = 1, header = FALSE)[[1]]
  data.table(jahr = j, vorhanden = sum(ana_subs %in% nm),
             fehlend = sum(!ana_subs %in% nm))
}))
print(panel_check)
cat(sprintf("\nPanel in jedem Jahr identisch: %s\n\n",
            all(panel_check$vorhanden == length(ana_subs))))

TOP_PC  <- 10L
politik <- !lifestyle
sub_lab <- panel[!is.na(politik)]
lab_vec <- politik[!is.na(politik)]

pc_check <- rbindlist(lapply(JAHRE, function(j) {
  v <- fread(vek_datei(j), skip = 1, header = FALSE)
  setnames(v, 1, "sub")

  # Störgröße: laden die ersten Hauptkomponenten auf die Rohnorm
  va       <- v[sub %in% ana_subs]
  Ra       <- as.matrix(va[, -1])
  raw_norm <- sqrt(rowSums(Ra^2))
  Ua       <- svd(scale(Ra / raw_norm, center = TRUE, scale = FALSE),
                  nu = TOP_PC, nv = 0)$u
  cor_norm <- abs(apply(Ua, 2, function(p) cor(p, log(raw_norm))))

  # Politische Trennung auf dem vollen gelabelten Panel
  vf   <- v[sub %in% sub_lab]
  lab  <- as.numeric(lab_vec[match(vf$sub, sub_lab)])
  Mf   <- as.matrix(vf[, -1])
  Mf   <- Mf / sqrt(rowSums(Mf^2))
  sf   <- svd(scale(Mf, center = TRUE, scale = FALSE))
  lamf <- sf$d^2 / sum(sf$d^2)
  cor_pol <- abs(apply(sf$u, 2, function(p) cor(p, lab)))
  best <- which.max(cor_pol)

  data.table(jahr = j,
             pc1_freq_cor   = cor_norm[1],
             max_freq_cor   = max(cor_norm),
             politik_PC     = best,
             politik_cor    = cor_pol[best],
             politik_PC_var = lamf[best])
}))
print(pc_check)

cat("\nKendalls tau:\n")
trend(pc_check, c("pc1_freq_cor", "max_freq_cor", "politik_PC",
                  "politik_cor", "politik_PC_var"))

readr::write_csv(pc_check, file.path(OUT_DIR, "ff1_pc_check.csv"))

    jahr vorhanden fehlend
   <int>     <int>   <int>
1:  2016     14412       0
2:  2017     14412       0
3:  2018     14412       0
4:  2019     14412       0
5:  2020     14412       0
6:  2021     14412       0
7:  2022     14412       0
8:  2023     14412       0
9:  2024     14412       0

Panel in jedem Jahr identisch: TRUE

    jahr pc1_freq_cor max_freq_cor politik_PC politik_cor politik_PC_var
   <int>        <num>        <num>      <int>       <num>          <num>
1:  2016   0.05365738    0.3823536         10   0.2232074     0.01459762
2:  2017   0.00825093    0.3585743         10   0.3218902     0.01471260
3:  2018   0.02400202    0.4021853         10   0.2895022     0.01493391
4:  2019   0.08918275    0.3492535         10   0.2907605     0.01547452
5:  2020   0.18880120    0.3895660         10   0.2423695     0.01573883
6:  2021   0.21166850    0.3920044          9   0.1867171     0.01654288
7:  2022   0.23862616    0.3784622          9   0.2366703     0.01689197
8:  2023

## Schwellengraph als andere Kantenregel

In [9]:
load_norm <- function(jahr, subs) {
  v <- fread(vek_datei(jahr), skip = 1, header = FALSE)
  setnames(v, 1, "sub")
  v <- v[sub %in% subs]
  M <- as.matrix(v[, -1])
  M <- M[, colSums(is.na(M)) < nrow(M), drop = FALSE]
  rownames(M) <- v$sub
  M / sqrt(rowSums(M^2))
}

# Schwelle so wählen, dass 2016 im Mittel die gewünschte Nachbarzahl erreicht
calibrate_tau <- function(M, ziel_grad, S = 3e6) {
  n  <- nrow(M)
  a  <- sample(n, S, replace = TRUE)
  b  <- sample(n, S, replace = TRUE)
  ok <- a != b
  cs <- rowSums(M[a[ok], ] * M[b[ok], ])
  frac <- (ziel_grad * n / 2) / (n * (n - 1) / 2)
  as.numeric(quantile(cs, 1 - frac))
}

# Kantenliste blockweise, damit die volle Ähnlichkeitsmatrix nie im Speicher steht
threshold_edges <- function(M, tau, block = 2000) {
  n <- nrow(M)
  out <- list()
  for (a in seq(1, n, by = block)) {
    b <- min(a + block - 1, n)
    S <- M[a:b, , drop = FALSE] %*% t(M)
    idx <- which(S > tau, arr.ind = TRUE)
    if (nrow(idx) == 0) next
    zeile <- (a:b)[idx[, 1]]
    spalte <- idx[, 2]
    keep <- spalte > zeile
    if (!any(keep)) next
    out[[length(out) + 1]] <- data.table(
      from = zeile[keep], to = spalte[keep], weight = S[idx[keep, , drop = FALSE]])
  }
  el <- rbindlist(out)
  el[, `:=`(from = rownames(M)[from], to = rownames(M)[to])]
  el[]
}

schwellen_graph <- function(M, tau) {
  graph_from_data_frame(threshold_edges(M, tau), directed = FALSE,
                        vertices = data.frame(name = rownames(M)))
}

In [10]:
set.seed(42)
ZIEL_GRAD <- 23
M16 <- load_norm(2016, ana_subs)
tau <- calibrate_tau(M16, ZIEL_GRAD)
cat(sprintf("Schwelle fuer mittlere Nachbarzahl %d in 2016: %.4f\n", ZIEL_GRAD, tau))
rm(M16)
invisible(gc(verbose = FALSE))

Schwelle fuer mittlere Nachbarzahl 23 in 2016: 0.6869


In [11]:
thr <- rbindlist(lapply(JAHRE, function(j) {
  M  <- load_norm(j, ana_subs)
  n  <- nrow(M)
  a  <- sample(n, 2e5, TRUE)
  b  <- sample(n, 2e5, TRUE)
  ok <- a != b
  mean_cos <- mean(rowSums(M[a[ok], ] * M[b[ok], ]))
  g  <- schwellen_graph(M, tau)
  wt <- cluster_walktrap(g, weights = E(g)$weight)
  sz <- sort(as.integer(table(membership(wt))), decreasing = TRUE)
  N  <- vcount(g)
  data.table(jahr = j, n = n, n_edges = ecount(g),
             mean_degree = 2 * ecount(g) / n, density = edge_density(g),
             mean_cos = mean_cos, isolierte = sum(degree(g) == 0),
             komp = components(g)$no, module = length(wt),
             Q = modularity(g, membership(wt), weights = E(g)$weight),
             top1 = sz[1] / N, top2 = sum(sz[1:2]) / N,
             rosenbluth = ineq::conc(sz, type = "Rosenbluth"))
}))
print(thr)

cat("\nKendalls tau:\n")
trend(thr, c("mean_degree", "density", "mean_cos", "module", "Q",
             "top2", "rosenbluth", "isolierte"))

readr::write_csv(thr, file.path(OUT_DIR, "ff1_threshold_roh.csv"))

    jahr     n n_edges mean_degree      density  mean_cos isolierte  komp
   <int> <int>   <num>       <num>        <num>     <num>     <int> <num>
1:  2016 14412  171302    23.77213 0.0016495826 0.2596869      5414  6063
2:  2017 14412   89050    12.35776 0.0008575225 0.2496633      5063  5733
3:  2018 14412   80426    11.16098 0.0007744762 0.2440555      4423  5051
4:  2019 14412   83942    11.64890 0.0008083342 0.2438942      3414  3939
5:  2020 14412   94370    13.09603 0.0009087524 0.2473333      2634  3081
6:  2021 14412  100670    13.97030 0.0009694194 0.2486011      2322  2723
7:  2022 14412  111927    15.53247 0.0010778206 0.2479005      2090  2421
8:  2023 14412  120686    16.74799 0.0011621669 0.2512794      1878  2165
9:  2024 14412  119590    16.59589 0.0011516128 0.2478380      1884  2110
   module         Q       top1       top2   rosenbluth
    <int>     <num>      <num>      <num>        <num>
1:   7090 0.5609168 0.05648071 0.10699417 0.0002674844
2:   6671 0.6969235 0

In [12]:
kern_masse <- function(jahr, tau, min_size = 5) {
  M  <- load_norm(jahr, ana_subs)
  g0 <- schwellen_graph(M, tau)
  komp_alle <- components(g0)$no
  iso <- sum(degree(g0) == 0)

  g  <- delete_vertices(g0, V(g0)[degree(g0) == 0])
  cp <- components(g)
  g  <- induced_subgraph(g, V(g)[cp$membership == which.max(cp$csize)])

  wt <- cluster_walktrap(g, weights = E(g)$weight)
  sz <- sort(as.integer(table(membership(wt))), decreasing = TRUE)
  data.table(jahr = jahr, n_core = vcount(g), iso = iso, n_edges = ecount(g),
             komp = komp_alle,
             module_all = length(wt),
             module_min = sum(sz >= min_size),
             Q = modularity(g, membership(wt), weights = E(g)$weight),
             top2 = sum(sz[1:2]) / vcount(g),
             rosenbluth = ineq::conc(sz, type = "Rosenbluth"))
}

SPALTEN_KERN <- c("jahr", "n_core", "n_edges", "komp", "module_all",
                  "module_min", "Q", "top2", "rosenbluth")

thr_clean <- rbindlist(lapply(JAHRE, kern_masse, tau = tau))
print(thr_clean[, ..SPALTEN_KERN])

cat("\nKendalls tau:\n")
trend(thr_clean, c("n_core", "module_all", "module_min", "Q", "top2", "rosenbluth"))

readr::write_csv(thr_clean[, ..SPALTEN_KERN],
                 file.path(OUT_DIR, "ff1_threshold_clean.csv"))

    jahr n_core n_edges  komp module_all module_min         Q      top2
   <int>  <num>   <num> <num>      <int>      <int>     <num>     <num>
1:  2016   6252  167546  6063        806        261 0.5437295 0.2466411
2:  2017   6415   84789  5733        743        295 0.6726303 0.1820733
3:  2018   7071   74715  5051        815        340 0.6494894 0.1622119
4:  2019   8784   80250  3939        835        381 0.7442776 0.1397996
5:  2020   9855   90476  3081        774        401 0.7798796 0.1710807
6:  2021  10366   97412  2723        830        425 0.8018112 0.1515532
7:  2022  10992  109418  2421        933        446 0.7991093 0.1321870
8:  2023  11494  118798  2165        828        432 0.7962128 0.1332869
9:  2024  11676  117722  2110        734        420 0.7903559 0.1219596
    rosenbluth
         <num>
1: 0.003762987
2: 0.003764360
3: 0.003304939
4: 0.003286714
5: 0.003744322
6: 0.003501731
7: 0.003189143
8: 0.003689506
9: 0.004272022

Kendalls tau:
  n_core         tau = +1.00

In [13]:
sweep <- rbindlist(lapply(c(0.55, 0.60, 0.65, 0.70), function(tt) {
  tab <- rbindlist(lapply(JAHRE, kern_masse, tau = tt))
  data.table(tau = tt,
             iso2016      = tab$iso[1],
             ncore_16_24  = sprintf("%d -> %d", tab$n_core[1], tab$n_core[nrow(tab)]),
             modmin_16_24 = sprintf("%d -> %d", tab$module_min[1], tab$module_min[nrow(tab)]),
             Q_16_24      = sprintf("%.2f -> %.2f", tab$Q[1], tab$Q[nrow(tab)]),
             tau_module_min = cor(tab$jahr, tab$module_min, method = "kendall"),
             tau_Q          = cor(tab$jahr, tab$Q,          method = "kendall"))
}))
print(as.data.frame(sweep))

cat(sprintf("\ntau_module_min ueber alle Schwellen: min %+.2f | max %+.2f\n",
            min(sweep$tau_module_min), max(sweep$tau_module_min)))
cat(sprintf("tau_Q          ueber alle Schwellen: min %+.2f | max %+.2f\n",
            min(sweep$tau_Q), max(sweep$tau_Q)))

readr::write_csv(sweep, file.path(OUT_DIR, "ff1_threshold_sweep.csv"))

   tau iso2016    ncore_16_24 modmin_16_24      Q_16_24 tau_module_min
1 0.55     649 13671 -> 14133    345 -> 99 0.58 -> 0.69    -1.00000000
2 0.60    1959 11922 -> 13753   410 -> 252 0.59 -> 0.71    -0.72222222
3 0.65    3815  8928 -> 12836   349 -> 351 0.58 -> 0.76    -0.08451543
4 0.70    5930  5393 -> 11056   235 -> 438 0.52 -> 0.80     0.94444444
      tau_Q
1 0.8333333
2 0.7222222
3 0.6666667
4 0.6666667

tau_module_min ueber alle Schwellen: min -1.00 | max +0.94
tau_Q          ueber alle Schwellen: min +0.67 | max +0.83


In [14]:
sessionInfo()

R version 4.5.1 (2025-06-13 ucrt)
Platform: x86_64-w64-mingw32/x64
Running under: Windows 11 x64 (build 26200)

Matrix products: default
  LAPACK version 3.12.1

locale:
[1] LC_COLLATE=German_Germany.utf8  LC_CTYPE=German_Germany.utf8   
[3] LC_MONETARY=German_Germany.utf8 LC_NUMERIC=C                   
[5] LC_TIME=German_Germany.utf8    

time zone: Europe/Berlin
tzcode source: internal

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] lubridate_1.9.4   forcats_1.0.0     stringr_1.5.1     dplyr_1.1.4      
 [5] purrr_1.1.0       readr_2.1.5       tidyr_1.3.1       tibble_3.3.0     
 [9] ggplot2_3.5.2     tidyverse_2.0.0   dbscan_1.2.2      ineq_0.2-13      
[13] igraph_2.3.2      data.table_1.17.8

loaded via a namespace (and not attached):
 [1] bit_4.6.0          gtable_0.3.6       crayon_1.5.3       compiler_4.5.1    
 [5] tidyselect_1.2.1   Rcpp_1.1.0         parallel_4.5.1     scales_1.4.0      
 [9] 